# cascade: oil sensitivity statistics

This notebook measures how weekly oil-price moves (Brent) show up in the weekly returns of the 14 MVP tickers, and how that reaches the three demo portfolios. All the work happens in `compute/stats.py`; each cell below calls one function.

It runs in SageMaker AI and uses the notebook's IAM role to read `s3://$BUCKET/raw/` and write `s3://$BUCKET/processed/`. Set the environment variable `BUCKET` first. Simulation only, not advice.

In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, so `compute` is importable
from compute import stats

BUCKET = os.environ["BUCKET"]
BUCKET

## Step 1. Load the raw files

Reads the price files (the starter file plus the extra ETFs), Brent oil prices, the USD/CAD exchange rate, the four government bond yields and the 10-year breakeven inflation rate from S3. Where a folder has dated snapshots, it picks the newest one.

In [ ]:
raw = stats.load_raw(BUCKET)
{name: df.shape for name, df in raw.items()}

## Step 2. Build weekly returns

Keeps the 14 MVP tickers and turns daily data into weekly data, using the last available value each Friday. Prices, Brent and USD/CAD become weekly log returns. Bond yields and breakeven inflation become weekly changes in percentage points. US-dollar tickers are converted to Canadian dollars by adding the weekly change in USD/CAD. Weeks without a Brent return are dropped.

In [ ]:
data = stats.build_weekly(raw)
print(data.ret_cad.shape, data.ret_cad.index.min().date(), "to", data.ret_cad.index.max().date())
data.ret_cad.tail(3)

## Step 3. Sensitivities

For each ticker, estimates how much its weekly CAD return moves when Brent moves: a simple version, a version that also controls for the overall market, a version that leaves out 2022, separate betas for up and down weeks, and a test of how often the sign is right on the biggest oil moves after 2024. Ranges are 95% ranges and use standard errors that allow for correlation over time.

In [ ]:
sens = stats.sensitivities(data)
sens

## Step 4. Rolling beta

Re-estimates the simple beta over a sliding 52-week window for every ticker, to show whether the relationship is stable or drifts over time.

In [ ]:
rolling = stats.rolling_betas(data)
rolling.tail()

## Step 5. Links in the transmission chains

Measures each step of the indirect routes: oil to bond yields, oil to the exchange rate, oil to breakeven inflation, and then yields or the exchange rate to the bond ETFs and equity ETFs they are expected to drive.

In [ ]:
links = stats.links(data)
links

## Step 6. Chain reconciliation

Multiplies the link betas along each chain (for example oil to US 10-year yield to TLT) and compares the result with the direct oil beta. The share explained shows how much of the direct effect the chain accounts for.

In [ ]:
chains = stats.chain_reconciliation(links, sens)
chains

## Step 7. Duration check

For a 10% rise in Brent, works out how far yields should move, then what bond duration implies for the ETF price (TLT, XBB.TO, ZST.TO), and compares it with the move actually measured. Durations are constants at the top of `stats.py`.

In [ ]:
duration = stats.duration_check(links, sens)
duration

## Step 8. Persona holdings and portfolio exposure

Loads the holdings of the three demo investors (robert, amira, mateo) and multiplies each weight by that holding's oil beta. Adding the contributions gives each portfolio's oil beta.

In [ ]:
holdings = stats.persona_holdings()
exposure = stats.portfolio_exposure(holdings, sens)
stats.portfolio_betas(exposure)

## Step 9. Scenarios

Applies oil shocks of -30%, -15%, +15% and +30% to every persona and every ticker, and shows the estimated price impact with a low and high end taken from the beta range.

In [ ]:
scenarios = stats.scenarios(sens, exposure)
scenarios[scenarios["subject_type"] == "persona"]

## Step 10. Alternatives

For each swap in the data contract (for example robert's TLT for ZST.TO), recalculates the portfolio beta after the swap and compares the +30% oil impact before and after.

In [ ]:
alternatives = stats.alternatives(holdings, sens)
alternatives

## Step 11. Sanity checks and write to S3

First checks two results that must hold before anything is published: Exxon (XOM) reacts positively and significantly to oil, and the long-bond ETF (TLT) reacts negatively. Then writes every table as Parquet to `processed/<table>/part-0.parquet`, plus `processed/_summary/summary.md` with row counts and portfolio betas. Existing files are overwritten.

In [ ]:
s = sens.set_index("ticker")
assert s.loc["XOM", "beta_simple"] > 0 and s.loc["XOM", "p_simple"] < 0.05, "XOM check failed"
assert s.loc["TLT", "beta_simple"] < 0, "TLT check failed"

tables = {
    "sensitivities": sens,
    "rolling": rolling,
    "links": links,
    "chain_reconciliation": chains,
    "duration_check": duration,
    "persona_holdings": holdings,
    "portfolio_exposure": exposure,
    "scenarios": scenarios,
    "alternatives": alternatives,
}
stats.write_outputs(tables, BUCKET)